<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-08-tool-use-and-mcp/lesson-8.2-designing-tool-schemas/notebooks/exercises-8.2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 8.2 — Designing Tool Schemas
Netsetos GenAI Course v2.0 | Module 8: AI Agents

Strict mode compatibility, tool design principles, description engineering, nesting, versioning, MCP mapping, India schemas


In [ ]:
print('Module 8: AI Agents & MCP')
print('Lesson 8.2: Designing Tool Schemas')


## Ex 1: Strict Mode Compatibility Matrix


In [ ]:
print('Strict mode compatibility across providers:')
print()
for feat, openai, claude, gemini in [
    ('additionalProperties:false','REQUIRED','Recommended','Unreliable'),
    ('Optional fields','["string","null"] + required','Omit from required','optionalProperties'),
    ('anyOf','✅ (not at root)','✅','❌'),
    ('oneOf','❌ REJECTED','Limited','❌'),
    ('default keyword','❌ Ignored','❌','❌'),
    ('minItems/maxItems','❌ Hint only','❌','❌'),
    ('Max nesting','5 levels','Grammar limit','Rejects deep'),
    ('input_examples','❌','✅ UNIQUE','❌'),
]: print(f'  {feat:25s}: {openai:25s} | {claude:20s} | {gemini}')
print()
print('Portable subset: type, properties, required, enum, description, additionalProperties:false')


## Ex 2: Tool Design Principles


In [ ]:
print('Tool design principles:')
print()
for principle, rule, example in [
    ('Naming','verb_noun, consistent verbs','get_weather, create_order, search_products'),
    ('Granularity','5-20 tools optimal (~19 = sweet spot)','55-70% improvement vs full catalog'),
    ('Atomic','Split by side-effect profile','Separate: create_user, update_user, delete_user'),
    ('Composable','Combine sequential reads','Combined: get_customer_context (was 3 tools)'),
    ('Idempotency','idempotency_key for all writes','Prevents double-charge on agent retry'),
    ('MCP annotations','readOnlyHint, destructiveHint, idempotentHint','Risk tiers: A (approve) B (recommend) C (auto)'),
]: print(f'  {principle:15s}: {rule}')
   print(f'  {"":15s}  Example: {example}')
   print()


## Ex 3: Description Engineering


In [ ]:
print('Description engineering — the highest-ROI optimization:')
print()
good = '''Get the CURRENT weather conditions for a specific city.
Returns: temperature (Celsius/Fahrenheit), humidity (%),
wind speed (km/h), and conditions (sunny/cloudy/rainy/snowy).
Use when: user asks about current weather, what to wear,
or whether to bring an umbrella.
Do NOT use for: historical weather, climate data,
air quality, forecasts beyond 24 hours, or pollen counts.
Example: get_weather(location="Mumbai", unit="celsius")'''
bad = 'Weather tool'
print('BAD description (causes over-triggering):')
print(f'  "{bad}"')
print()
print('GOOD description (30%+ accuracy improvement):')
for line in good.split('\n'):
    print(f'  {line}')


## Ex 4: Nesting vs Flattening


In [ ]:
print('Nested vs flat schemas — accuracy comparison:')
print()
print('NESTED (3 levels) — GPT-4o: 28% accuracy:')
print('  {"user": {"address": {"city": "Mumbai", "state": "MH"}}}')
print()
print('FLAT (0 levels) — GPT-4o: 85%+ accuracy:')
print('  {"user__address__city": "Mumbai", "user__address__state": "MH"}')
print()
print('SHALLOW NESTED (1 level) — recommended for cohesive groups:')
print('  {"location": {"city": "Mumbai", "state": "MH"}}')
print()
print('Rule: flatten beyond 2 levels. Reserve nesting for')
print('semantically cohesive groups like address, coordinates.')


## Ex 5: Schema Versioning


In [ ]:
print('Schema versioning strategy:')
print()
for change, breaking, note in [
    ('Rename parameter','ALWAYS breaking','Old agents fail silently'),
    ('Change type','ALWAYS breaking','String→int breaks parsing'),
    ('Optional→required','ALWAYS breaking','Old calls missing field'),
    ('Add optional param (Claude/Gemini)','Non-breaking','Claude allows genuine optionals'),
    ('Add ANY field (OpenAI strict)','BREAKING!','additionalProperties:false rejects it'),
    ('Improve description','Non-breaking','Improves accuracy, no breakage'),
    ('Add enum value','Non-breaking (if under limits)','OpenAI: max 500 values'),
]: print(f'  {change:35s}: {"⚠️ " + breaking if "breaking" in breaking.lower() else "✅ " + breaking}')
   print(f'  {"":35s}  {note}')
   print()


## Ex 6: MCP Schema Mapping


In [ ]:
print('Function calling → MCP schema mapping:')
print()
print('OpenAI:')
print('  {"type": "function", "function": {"name": "X", "parameters": SCHEMA}}')
print()
print('Claude:')
print('  {"name": "X", "input_schema": SCHEMA}')
print()
print('MCP:')
print('  {"name": "X", "inputSchema": SCHEMA}')
print()
print('Mapping: parameters = input_schema = inputSchema = SAME JSON Schema')
print()
print('MCP primitives:')
print('  Tools: model-controlled, may have side effects')
print('  Resources: app-controlled, read-only data by URI')
print('  Prompts: user-controlled, parameterized templates')


## Ex 7: India Schema Patterns


In [ ]:
import unicodedata

print('Unicode NFC normalization for Indic scripts:')
print()
# Devanagari nukta example
composed = '\u0958'  # क़ (single codepoint)
decomposed = '\u0915\u093C'  # क + ़
nfc_composed = unicodedata.normalize('NFC', composed)
nfc_decomposed = unicodedata.normalize('NFC', decomposed)
print(f'  Composed (U+0958): {repr(composed)}')
print(f'  Decomposed (U+0915+U+093C): {repr(decomposed)}')
print(f'  NFC(composed): {repr(nfc_composed)}')
print(f'  NFC(decomposed): {repr(nfc_decomposed)}')
print(f'  Equal after NFC: {nfc_composed == nfc_decomposed}')
print()
print('Indian identifier validation patterns:')
for name, pattern in [
    ('PAN','[A-Z]{5}[0-9]{4}[A-Z]{1}'),
    ('Aadhaar','[2-9][0-9]{11} (Verhoeff check)'),
    ('GSTIN','[0-9]{2}[A-Z]{5}[0-9]{4}[A-Z]{1}[1-9A-Z]{1}Z[0-9A-Z]{1}'),
    ('UPI VPA','[a-zA-Z0-9._-]+@[a-zA-Z]+'),
    ('Mobile','+91[6-9][0-9]{9}'),
    ('Pincode','[1-9][0-9]{5}'),
]: print(f'  {name:10s}: {pattern}')


## Ex 8: 5-Layer Testing


In [ ]:
print('5-layer testing strategy for tool schemas:')
print()
for layer, what, tool in [
    ('1. Schema validation','Provider-specific validators','FastMCP strict_input_validation=True'),
    ('2. Quality scoring','Tool selection accuracy','DeepEval ToolCorrectnessMetric'),
    ('3. Mock execution','Reasoning without real APIs','Scenario framework (langwatch.ai)'),
    ('4. Edge cases','Null/empty/Unicode/pagination','BFCL V2: 875 irrelevance cases'),
    ('5. Regression','Golden examples in CI/CD','20-30+ examples, 5% drop = regression'),
]: print(f'  {layer:25s}')
   print(f'    What: {what}')
   print(f'    Tool: {tool}')
   print()


---
## Recap
Schema design IS the difference between demo and production. Use the portable subset for cross-provider compatibility. Description engineering is the highest-ROI optimization (30%+ accuracy). Flatten beyond 2 levels (28% accuracy gap). ~19 tools optimal (55-70% improvement). Version schemas with CI/CD golden tests. MCP mapping: parameters→inputSchema (same JSON Schema, different wrapper). For India: Unicode NFC normalization, Indian identifier validation, code-mixing support, Bhashini tool wrapping.
